# AMPS II · Week 11 — HW10
## Scalings and symmetry, manufactured solutions, the Nusselt number, $Nu(Ra)$, Picard, Rayleigh–Bénard and an LLM-generated benchmark script

**Due before Week 12 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong. The whole HW runs in a few minutes.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: scalings, the consistent flux, the symmetry of the cavity | 15 |
| 2 | Manufactured solution: convergence rates of the coupled system | 14 |
| 3 | The Nusselt number: mesh, grading and how it is computed | 14 |
| 4 | $Nu(Ra)$: the boundary-layer law | 13 |
| 5 | Picard: contraction against $Ra$, relaxation | 14 |
| 6 | Heated from below: two branches and the onset of convection | 15 |
| 7 | Debugging an LLM-generated benchmark script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run the cell once.

In [ ]:
%%writefile /tmp/fnx/bouss.py
"""bouss.py — Boussinesq natural convection in a box, DOLFINx 0.11 (AMPS II, Week 11).

Nondimensional with L and the thermal velocity kappa/L (the scaling of the de Vahl Davis benchmark):
    (u.grad) u = -grad p + Pr lap u + Pr Ra T e_y,   div u = 0,   u.grad T = lap T
Taylor-Hood (P2/P1) for (u, p) and P2 for T in one mixed space; Newton (monolithic) or Picard (segregated)."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh, geometry
from dolfinx.fem.petsc import NonlinearProblem
import dolfinx.fem.petsc as fp

comm = MPI.COMM_WORLD
# PETSc objects must be destroyed collectively: no automatic garbage collection, collect() at the same point on all ranks
gc.disable()
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-9, snes_stol=0.0, snes_max_it=25)

def box_mesh(N, L=1.0, H=1.0, grade=0.0, Ny=None):
    """[0, L] x [0, H] with N x Ny triangles; grade in [0, 1): cells clustered towards all walls (s -> s - grade sin(2 pi s)/(2 pi))"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [N, Ny or N], mesh.CellType.triangle)
    if grade > 0:
        x = msh.geometry.x
        for i, Li in ((0, L), (1, H)):
            s = x[:, i] / Li; x[:, i] = Li * (s - grade * np.sin(2 * np.pi * s) / (2 * np.pi))
    return msh

def _box(msh):
    x = msh.geometry.x
    return comm.allreduce(x[:, 0].max(), op=MPI.MAX), comm.allreduce(x[:, 1].max(), op=MPI.MAX)

class Convection:
    count = 0
    def __init__(self, msh, Ra=1e3, Pr=0.71, heated="side", deg=2, mms=None, guess="conduction", roll=0.0):
        """heated "side": T = 1 at x = 0, T = 0 at x = L, top and bottom adiabatic (de Vahl Davis);
        heated "below": T = 1 at y = 0, T = 0 at y = H, sides adiabatic (Rayleigh-Benard). No slip everywhere; p = 0 at (0, 0).
        mms: dict of UFL expressions u, p, T of the coordinates -> source terms and Dirichlet data from them (all walls).
        guess: "conduction" (linear T, u = 0) or "zero"; roll: amplitude of an added convection roll (u and T), see reset()."""
        collect()
        self.msh = msh; c = msh.basix_cell(); self.deg = deg; self.heated = heated
        self.L, self.H = _box(msh)
        el = basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, deg, shape=(2,)), basix.ufl.element("Lagrange", c, deg - 1),
                                      basix.ufl.element("Lagrange", c, deg)])
        W = self.W = fem.functionspace(msh, el); self.w = fem.Function(W)
        self.Ra, self.Pr = fem.Constant(msh, float(Ra)), fem.Constant(msh, float(Pr))
        self.ey = ufl.as_vector([0.0, 1.0]); self.n = ufl.FacetNormal(msh)
        u, p, T = ufl.split(self.w); v, q, s = ufl.TestFunctions(W)
        self.F = (ufl.inner(ufl.grad(u) * u, v) + self.Pr * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)
                  - self.Pr * self.Ra * T * ufl.dot(self.ey, v) + ufl.dot(u, ufl.grad(T)) * s + ufl.dot(ufl.grad(T), ufl.grad(s))) * ufl.dx
        self.mms = mms
        if mms is not None:
            ue, pe, Te = mms["u"], mms["p"], mms["T"]
            f = ufl.grad(ue) * ue - self.Pr * ufl.div(ufl.grad(ue)) + ufl.grad(pe) - self.Pr * self.Ra * Te * self.ey
            g = ufl.dot(ue, ufl.grad(Te)) - ufl.div(ufl.grad(Te))
            self.F -= (ufl.dot(f, v) + g * s) * ufl.dx
        self.V, _ = W.sub(0).collapse(); self.Q, _ = W.sub(1).collapse(); self.S, self.mapS = W.sub(2).collapse()
        self.bcs = self._bcs()
        Convection.count += 1
        self.pb = NonlinearProblem(self.F, self.w, bcs=self.bcs, petsc_options_prefix=f"cv{Convection.count}_", petsc_options=NEWTON)
        self.pb.solver.setConvergenceHistory(reset=True)
        self.reset(guess, roll)

    def _bcs(self):
        W, fdim = self.W, 1
        allb = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.full(x.shape[1], True))
        pd = fem.locate_dofs_geometrical((W.sub(1), self.Q), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0))
        if self.mms is not None:                                        # exact data on every wall, exact pressure at the corner
            uD, pD, TD = fem.Function(self.V), fem.Function(self.Q), fem.Function(self.S)
            for fun, e in ((uD, self.mms["u"]), (pD, self.mms["p"]), (TD, self.mms["T"])):
                fun.interpolate(fem.Expression(e, fun.function_space.element.interpolation_points))
            self.ax = None
            return [fem.dirichletbc(uD, fem.locate_dofs_topological((W.sub(0), self.V), fdim, allb), W.sub(0)),
                    fem.dirichletbc(pD, pd, W.sub(1)),
                    fem.dirichletbc(TD, fem.locate_dofs_topological((W.sub(2), self.S), fdim, allb), W.sub(2))]
        ax, top = (0, self.L) if self.heated == "side" else (1, self.H)
        hot = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.isclose(x[ax], 0))
        cold = mesh.locate_entities_boundary(self.msh, fdim, lambda x: np.isclose(x[ax], top))
        self.hot, self.cold, self.ax = hot, cold, ax
        tags = np.concatenate([hot, cold]); vals = np.r_[np.full(len(hot), 1), np.full(len(cold), 2)].astype(np.int32); o = np.argsort(tags)
        self.ds = ufl.Measure("ds", domain=self.msh, subdomain_data=mesh.meshtags(self.msh, fdim, tags[o], vals[o]))
        Th = fem.Function(self.S); Th.x.array[:] = 1.0
        return [fem.dirichletbc(fem.Function(self.V), fem.locate_dofs_topological((W.sub(0), self.V), fdim, allb), W.sub(0)),
                fem.dirichletbc(fem.Function(self.Q), pd, W.sub(1)),
                fem.dirichletbc(Th, fem.locate_dofs_topological((W.sub(2), self.S), fdim, hot), W.sub(2)),
                fem.dirichletbc(fem.Function(self.S), fem.locate_dofs_topological((W.sub(2), self.S), fdim, cold), W.sub(2))]

    def reset(self, guess="conduction", roll=0.0):
        """initial guess: conduction (T linear between the walls, u = 0) or zero; plus a roll filling the box:
        u = 10 roll * curl(sin^2(pi x/L) sin^2(pi y/H)) / pi, T += 0.3 roll cos(pi x/L) sin(pi y/H) (warm fluid rising at x = 0)"""
        self.w.x.array[:] = 0.0
        if guess == "conduction" and self.mms is None:
            tf = fem.Function(self.S); top = self.L if self.ax == 0 else self.H; ax = self.ax
            tf.interpolate(lambda x: 1 - x[ax] / top); self.w.x.array[self.mapS] = tf.x.array
        if roll:
            a, L, H = 10.0 * roll, self.L, self.H
            Vc, mV = self.W.sub(0).collapse(); uf = fem.Function(Vc)
            uf.interpolate(lambda x: np.vstack([-a * np.sin(np.pi * x[0] / L) ** 2 * np.sin(2 * np.pi * x[1] / H),
                                                a * np.sin(2 * np.pi * x[0] / L) * np.sin(np.pi * x[1] / H) ** 2]))
            self.w.x.array[mV] = uf.x.array
            tf = fem.Function(self.S); tf.interpolate(lambda x: 0.3 * roll * np.cos(np.pi * x[0] / L) * np.sin(np.pi * x[1] / H))
            self.w.x.array[self.mapS] += tf.x.array
        self.w.x.scatter_forward()

    def solve(self, Ra=None):
        """Newton on the mixed (u, p, T) system from the current state; returns (SNES reason, iterations)"""
        if Ra is not None:
            self.Ra.value = Ra
        self.pb.solve(); r = self.pb.solver
        return int(r.getConvergedReason()), int(r.getIterationNumber())

    def history(self):
        """residual norms of the last Newton solve"""
        return np.array(self.pb.solver.getConvergenceHistory()[0])

    def continuation(self, Ras):
        """Newton at each Ra, starting from the previous solution; returns [(Ra, reason, iterations)]"""
        return [(Ra, *self.solve(Ra)) for Ra in Ras]

    # ── quantities ──────────────────────────────────────────────────
    def nusselt(self):
        """(Nu_hot, Nu_cold) from the wall gradient: mean heat flux into the fluid at the hot wall / out of it at the cold wall"""
        T = ufl.split(self.w)[2]
        f = lambda t: comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(ufl.grad(T), self.n) * self.ds(t))), op=MPI.SUM)
        wall = self.H if self.ax == 0 else self.L
        return f(1) / wall, -f(2) / wall

    def nusselt_flux(self):
        """(Nu_hot, Nu_cold) as a consistent (residual) flux: the weak energy equation tested with a function that is 1 on one wall"""
        u, p, T = ufl.split(self.w); out = []
        for facets in (self.hot, self.cold):
            z = fem.Function(self.S); z.x.array[fem.locate_dofs_topological(self.S, 1, facets)] = 1.0; z.x.scatter_forward()
            out.append(comm.allreduce(fem.assemble_scalar(fem.form((ufl.dot(u, ufl.grad(T)) * z + ufl.dot(ufl.grad(T), ufl.grad(z))) * ufl.dx)), op=MPI.SUM))
        wall = self.H if self.ax == 0 else self.L
        return out[0] / wall, -out[1] / wall

    def errors(self):
        """L2 errors of u, p, T and H1-seminorm errors of u, T against the manufactured solution"""
        u, p, T = ufl.split(self.w); m = self.mms
        a = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM))
        du, dT = u - m["u"], T - m["T"]
        return dict(u=a(ufl.inner(du, du)), p=a((p - m["p"]) ** 2), T=a(dT ** 2), gu=a(ufl.inner(ufl.grad(du), ufl.grad(du))), gT=a(ufl.inner(ufl.grad(dT), ufl.grad(dT))))

    def probe(self, comp, P):
        """values at points P (n x 3) of comp 0, 1 = u, v; 2 = T; 3 = p — every rank gets all values"""
        fun = self.w.sub({0: 0, 1: 0, 2: 2, 3: 1}[comp]).collapse()
        nloc = self.msh.topology.index_map(2).size_local
        cand = geometry.compute_colliding_cells(self.msh, geometry.compute_collisions_points(geometry.bb_tree(self.msh, 2), P), P)
        out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
        for j in range(len(P)):
            own = [c for c in cand.links(j) if c < nloc]
            if own:
                idx.append(j); cells.append(own[0])
        if idx:
            val = np.asarray(fun.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
            out[idx] = val[:, comp] if comp < 2 else val[:, 0]; cnt[idx] = 1
        out = comm.allreduce(out, op=MPI.SUM); cnt = comm.allreduce(cnt, op=MPI.SUM)
        return out / np.maximum(cnt, 1)

    def line(self, comp, x0=None, y0=None, n=401):
        t = np.linspace(0, 1, n); P = np.zeros((n, 3))
        if x0 is not None:
            P[:, 0], P[:, 1] = x0, t * self.H
        else:
            P[:, 0], P[:, 1] = t * self.L, y0
        return t, self.probe(comp, P)

    def peak(self, comp, x0=None, y0=None, n=2001):
        """maximum of a component along a line and its position (parabola through the three best samples)"""
        t, v = self.line(comp, x0=x0, y0=y0, n=n); i = int(np.argmax(v))
        if 0 < i < n - 1:
            a, b, c = v[i - 1], v[i], v[i + 1]; s = 0.5 * (a - c) / (a - 2 * b + c)
            return b - 0.25 * (a - c) * s, t[i] + s * (t[1] - t[0])
        return v[i], t[i]

    def umax(self):
        """max |u| over the velocity dofs"""
        a = self.w.x.array[self.W.sub(0).collapse()[1]]
        return float(comm.allreduce(np.abs(a).max() if len(a) else 0.0, op=MPI.MAX))

class Picard:
    """segregated iteration: the flow (Navier-Stokes, Newton) with the last T, then the energy equation (linear) with that u"""
    count = 0
    def __init__(self, cv, relax=1.0):
        collect(); Picard.count += 1
        self.cv = cv; msh = cv.msh; c = msh.basix_cell(); d = cv.deg; self.relax = relax
        Wf = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, d, shape=(2,)), basix.ufl.element("Lagrange", c, d - 1)]))
        self.wf = fem.Function(Wf); self.T = fem.Function(cv.S)
        u, p = ufl.split(self.wf); v, q = ufl.TestFunctions(Wf)
        Ff = (ufl.inner(ufl.grad(u) * u, v) + cv.Pr * ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u)
              - cv.Pr * cv.Ra * self.T * ufl.dot(cv.ey, v)) * ufl.dx
        V, _ = Wf.sub(0).collapse(); Q, _ = Wf.sub(1).collapse()
        allb = mesh.locate_entities_boundary(msh, 1, lambda x: np.full(x.shape[1], True))
        bf = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((Wf.sub(0), V), 1, allb), Wf.sub(0)),
              fem.dirichletbc(fem.Function(Q), fem.locate_dofs_geometrical((Wf.sub(1), Q), lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0)), Wf.sub(1))]
        self.flow = NonlinearProblem(Ff, self.wf, bcs=bf, petsc_options_prefix=f"pf{Picard.count}_", petsc_options=NEWTON)
        Tt = ufl.TrialFunction(cv.S); s = ufl.TestFunction(cv.S); uu = ufl.split(self.wf)[0]
        self.Tn = fem.Function(cv.S); Th = fem.Function(cv.S); Th.x.array[:] = 1.0
        bT = [fem.dirichletbc(Th, fem.locate_dofs_topological(cv.S, 1, cv.hot)), fem.dirichletbc(fem.Function(cv.S), fem.locate_dofs_topological(cv.S, 1, cv.cold))]
        self.energy = fp.LinearProblem((ufl.dot(uu, ufl.grad(Tt)) * s + ufl.dot(ufl.grad(Tt), ufl.grad(s))) * ufl.dx, fem.Constant(msh, 0.0) * s * ufl.dx,
                                       bcs=bT, u=self.Tn, petsc_options_prefix=f"pe{Picard.count}_", petsc_options=LU)
        self.T.x.array[:] = cv.w.x.array[cv.mapS]                       # start from the state of cv

    def run(self, Ra, tol=1e-8, maxit=200):
        """returns (iterations, history of max|T_k - T_(k-1)|); an inf at the end: the flow solve failed"""
        self.cv.Ra.value = Ra; hist = []
        for k in range(maxit):
            self.flow.solve()
            if self.flow.solver.getConvergedReason() <= 0:
                return k + 1, hist + [np.inf]
            self.energy.solve()
            dT = comm.allreduce(np.abs(self.Tn.x.array - self.T.x.array).max(), op=MPI.MAX); hist.append(dT)
            self.T.x.array[:] = self.relax * self.Tn.x.array + (1 - self.relax) * self.T.x.array
            if dT < tol or not np.isfinite(dT) or dT > 1e3:
                return k + 1, hist
        return maxit, hist

# de Vahl Davis (1983), Pr = 0.71, scale kappa/L: mean Nu, u_max on x = 0.5 (value, y), v_max on y = 0.5 (value, x)
DVD = {1e3: dict(Nu=1.118, umax=(3.649, 0.813), vmax=(3.697, 0.178)), 1e4: dict(Nu=2.243, umax=(16.178, 0.823), vmax=(19.617, 0.119)),
       1e5: dict(Nu=4.519, umax=(34.73, 0.855), vmax=(68.59, 0.066)), 1e6: dict(Nu=8.800, umax=(64.63, 0.850), vmax=(219.36, 0.0379))}

## Q1 — Theory (15 pts)

**(a) (5)** Start from the dimensional Boussinesq equations $\rho_0(\mathbf u\cdot\nabla)\mathbf u=-\nabla p+\mu\Delta\mathbf u+\rho_0g\beta(T-T_0)\mathbf e_y$, $\nabla\cdot\mathbf u=0$, $\mathbf u\cdot\nabla T=\kappa\Delta T$.
Scale lengths by $L$, temperatures by $\Delta T$, and velocities (i) by $\kappa/L$, (ii) by $\nu/L$. Give the three dimensionless coefficients in both cases, and show $\mathbf u_\nu=\mathbf u_\kappa/Pr$ and that $Nu$ is the same.

**(b) (5)** Let $z_h$ be the P2 function that is 1 at the nodes of the hot wall and 0 at all other nodes. Show that for the exact solution $\int_\Omega(\mathbf u\cdot\nabla T\,z+\nabla T\cdot\nabla z)\,dx=\int_\text{hot}(-\partial_xT)\,ds$
for any smooth $z$ that is 1 on the hot wall and 0 on the cold wall, and explain why evaluating the left side with $T_h$, $z_h$ is more accurate than integrating $-\partial_xT_h$ on the wall.

**(c) (5)** Show that $(x,y)\mapsto(1-x,1-y)$, $T\mapsto1-T$, $\mathbf u\mapsto-\mathbf u$ (and $p$ suitably adjusted) maps a solution of the cavity problem to a solution. What follows for $Nu_\text{hot}$ and $Nu_\text{cold}$, and why is their agreement
not a test of a code?

> *your answer*

## Q2 — Manufactured solution: convergence rates of the coupled system (14 pts)

$\mathbf u=5\,\mathrm{curl}\,(\sin^2\pi x\,\sin^2\pi y)$, $p=\cos\pi x\sin\pi y$, $T=1-x+0.2\sin\pi x\cos\pi y$, $Ra=10^3$, $Pr=0.71$; `Convection(msh, mms=dict(u=..., p=..., T=...))` adds the sources and the boundary data.

**(a) (8)** Compute the $L^2$ errors of $\mathbf u,p,T$ and the $H^1$-seminorm errors of $\mathbf u,T$ for $N=8,16,32,64$ and the observed orders.

**(b) (6)** Which orders do you expect for Taylor–Hood P2/P1 with P2 temperature? Explain any observed order that is higher than expected.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) — errors and observed orders, N = 8, 16, 32, 64

> *your answer (Q2)*

## Q3 — The Nusselt number: mesh, grading and how it is computed (14 pts)

At $Ra=10^5$ (continuation $10^3\to10^4\to10^5$):

**(a) (8)** Compute $Nu$ from the wall gradient and as the consistent flux on uniform and graded (`grade=0.5`) meshes with $N=16,32,64$. Estimate the observed order of each sequence and Richardson-extrapolate the best one.

**(b) (6)** De Vahl Davis gives $Nu=4.519$ and claims about 1 % accuracy at the highest $Ra$. Is $4.519$ consistent with your extrapolation? Which of your 12 numbers would pass a "within 1 % of the benchmark" test, and what does that say about such tests?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) — Nu (gradient, flux) on uniform and graded meshes, N = 16, 32, 64; orders and Richardson

> *your answer (Q3)*

## Q4 — $Nu(Ra)$: the boundary-layer law (13 pts)

**(a) (7)** Compute $Nu$ (consistent flux) for $Ra=10^3,\,2\times10^3,\,5\times10^3,\dots,10^6$ (three per decade) by continuation on the graded $N=32$ mesh, and the local exponent $d\ln Nu/d\ln Ra$ between neighbours.

**(b) (6)** A laminar boundary layer predicts $Nu\propto Ra^{1/4}$. Does the local exponent approach $1/4$? Fit $Nu=C\,Ra^b$ on $[10^5,10^6]$ and comment on the use of such fits outside their range.

In [ ]:
%%fenicsx -np 2
# TODO: Q4 — Nu(Ra) by continuation, local exponents, a fit on [1e5, 1e6]

In [ ]:
import numpy as np, matplotlib.pyplot as plt
try:
    R = np.load("/tmp/fnx/nura.npy")
    fig, ax = plt.subplots(figsize=(5.5, 3.8)); ax.loglog(R[:, 0], R[:, 1], "o-", label="computed")
    ax.loglog(R[:, 0], R[-1, 1] * (R[:, 0] / R[-1, 0]) ** 0.25, "k--", lw=0.8, label=r"slope 1/4 through the last point")
    ax.set(xlabel="Ra", ylabel="Nu"); ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()
except FileNotFoundError:
    print("run the cell above first")

> *your answer (Q4)*

## Q5 — Picard: contraction against $Ra$, relaxation (14 pts)

**(a) (8)** On the graded $N=16$ mesh, run `Picard` from the conduction state for $Ra=10^3,\,3\times10^3,\,10^4,\,3\times10^4$ (at most 40 iterations) and estimate the asymptotic contraction factor $q(Ra)$. How does $1-q$ scale with $Ra$?

**(b) (6)** At $Ra=3\times10^4$ repeat with relaxation $\omega=0.5$ (`Picard(cv, relax=0.5)`). Explain the effect, and compare the cost with Newton (continuation from $10^4$).

In [ ]:
%%fenicsx -np 2
# TODO: Q5 — Picard contraction factors, relaxation at 3e4, Newton for comparison

> *your answer (Q5)*

## Q6 — Heated from below: two branches and the onset of convection (15 pts)

`Convection(msh, heated="below")`: $T=1$ at $y=0$, $T=0$ at $y=1$, adiabatic side walls. Uniform $N=32$.

**(a) (4)** Solve at $Ra=5000$ from the conduction state and from `guess="zero"`. Report reason, iterations, $Nu$, $\max|\mathbf u|$. Explain why Newton **must** return this solution from any start with $\mathbf u=0$.

**(b) (4)** Start from a convection roll (`roll=1.0`, and `roll=-1.0`) at $Ra=3000$ and continue to $5000$ and $10^4$; then repeat `roll=1.0` with steps of 500 up to 5000. Report $Nu$, $\max|\mathbf u|$, the iterations and the sign of $v$ at $(0.1,0.5)$ at every step. What happened in the step to 5000? Which of the solutions at $Ra=5000$ would you observe in an experiment?

**(c) (7)** Continue the roll branch down to $Ra=2620$ and estimate the onset $Ra_c$ from $\max|\mathbf u|^2$ (linear in $Ra$ near a pitchfork). Compare $N=16$ and $N=32$. Why does a direct solve "at $Ra_c$" not work?

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — conduction branch, roll branch, onset

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated benchmark script (15 pts)

An LLM was asked for "a DOLFINx 0.11 script for the de Vahl Davis benchmark at $Ra=10^4,10^5,10^6$, $Pr=0.71$, that verifies itself against the published Nusselt numbers". It returned the script below.

**(a) (3)** Run it as received (`-np 2`). What does it claim?

**(b) (7)** Find the defects. For each: what it does to the results, whether the script's own check can see it, and a test that exposes it.

**(c) (5)** Correct it (the module may be used) and verify: Nusselt numbers, velocity maxima **with their positions**, and the direction of the circulation.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): de Vahl Davis natural convection benchmark, Ra = 1e4, 1e5, 1e6, Pr = 0.71
import numpy as np, ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
comm = MPI.COMM_WORLD
Pr = 0.71
msh = mesh.create_unit_square(comm, 32, 32, mesh.CellType.triangle)
c = msh.basix_cell()
W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1),
                                                     basix.ufl.element("Lagrange", c, 2)]))
w = fem.Function(W); u, p, T = ufl.split(w); v, q, s = ufl.TestFunctions(W)
Ra = fem.Constant(msh, 1e4)
# nondimensionalised with the viscous velocity nu/L
F = (ufl.inner(ufl.grad(u) * u, v) + ufl.inner(ufl.grad(u), ufl.grad(v)) - p * ufl.div(v) - q * ufl.div(u) + Ra / Pr * T * v[1]
     + ufl.dot(u, ufl.grad(T)) * s + 1 / Pr * ufl.dot(ufl.grad(T), ufl.grad(s))) * ufl.dx
V, _ = W.sub(0).collapse(); S, mS = W.sub(2).collapse()
walls = mesh.locate_entities_boundary(msh, 1, lambda x: np.full(x.shape[1], True))
hot = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0.0)); cold = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 1.0))
one = fem.Function(S); one.x.array[:] = 1.0
bcs = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((W.sub(0), V), 1, walls), W.sub(0)),
       fem.dirichletbc(one, fem.locate_dofs_topological((W.sub(2), S), 1, hot), W.sub(2)),
       fem.dirichletbc(fem.Function(S), fem.locate_dofs_topological((W.sub(2), S), 1, cold), W.sub(2))]
problem = NonlinearProblem(F, w, bcs=bcs, petsc_options_prefix="dvd_",
                           petsc_options={"snes_type": "newtonls", "ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"})
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, np.sort(hot), np.ones(len(hot), dtype=np.int32)))
benchmark = {1e4: 2.243, 1e5: 4.519, 1e6: 8.800}
for Ra_ in benchmark:
    Ra.value = Ra_
    w.x.array[:] = 0.0                                                  # start from the conduction solution
    Tc = fem.Function(S); Tc.interpolate(lambda x: 1 - x[0]); w.x.array[mS] = Tc.x.array
    problem.solve()
    Nu = comm.allreduce(fem.assemble_scalar(fem.form(-T.dx(0) * ds(1))), op=MPI.SUM)
    umax = comm.allreduce(np.abs(w.sub(0).collapse().x.array).max(), op=MPI.MAX)
    err = abs(Nu - benchmark[Ra_]) / benchmark[Ra_]
    print(f"Ra = {Ra_:.0e}: Nu = {Nu:.3f} (benchmark {benchmark[Ra_]}, error {100 * err:.1f} %), max velocity {umax:.2f}  ->  {'PASS' if err < 0.05 else 'FAIL'}")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …